***all actor critics try to combine value base and policy gradient in one place.***

Advantage actor critics.

The Advantage Actor-Critic (A2C) algorithm combines the strengths of both policy-based and value-based methods in reinforcement learning.

key points to A2C:

Actor-Critic Architecture,

Policy Gradient Update

----------------

difference to vannila:

A2C — simply run N environments in parallel and average the gradients.


 in A2C, you have multiple actor environments — say N parallel environments — each running the same policy (same model weights) but with their own independent states.

output -> we collect info for multiple env, then based on that we do TD(0) and then we go with gradient dicent.

| Feature                | Vanilla Actor-Critic (VAC) | Advantage Actor-Critic (A2C)            |
| ---------------------- | -------------------------- | --------------------------------------- |
| **Parallel Envs**      | ❌ Single env               | ✅ Multiple vectorized envs              |
| **Update Frequency**   | Every step (online)        | After `n` steps from all envs (batch)   |
| **Advantage Estimate** | `A = TD(0)`                | `A = TD(n)` from rollout                |
| **Gradient Update**    | Per step                   | Batched across `(T × N_ENVS)` samples   |
| **Synchronization**    | Sequential, per step       | Synchronous, parallel rollout           |
| **Entropy Bonus**      | Optional, not always used  | Often included to encourage exploration |
| **Gradient Clipping**  | Optional                   | Commonly used in A2C                    |
| **Compute Efficiency** | Low (single-threaded)      | High (vectorized rollout, batch update) |
| **Stability**          | Often unstable             | Much more stable                        |




after each roll out we do TD(n), if rollout 2k based on number of worker you can see what is the n for your td.

------------------------------

A2C is something between ppo and vanila one.

| Feature                   | Vanilla Actor-Critic     | A2C                                | PPO                                        |
| ------------------------- | ------------------------ | ---------------------------------- | ------------------------------------------ |
| **Update frequency**      | Every single step        | After collecting N steps (e.g. 5)  | After collecting a large batch (e.g. 2048) |
| **Batch size**            | 1                        | `T × B` (e.g., 5 × 8 = 40)         | 2048 or larger                             |
| **Policy update**         | Once per step            | Once per rollout                   | **Multiple epochs per batch**              |
| **Log probs used**        | Immediate from current π | Recomputed from current π          | Compare current π to **old π** (ratio!)    |
| **Optimization loop**     | Immediate                | One `loss.backward()` and `step()` | 4–10 passes over the same batch            |
| **Clipping/Trust region** | ❌ No                     | ❌ No                               | ✅ Yes (`clip_ratio`, KL penalty, etc.)     |



In [1]:
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
from gymnasium.vector import SyncVectorEnv


In [2]:
# Create the CartPole Environment
env = gym.make('CartPole-v1')

In [3]:
ENV_ID          = "CartPole-v1"
N_ENVS          = 8                 # parallel actors
ROLLOUT_STEPS   = 5                 # T
TOTAL_UPDATES   = 1_000             # outer loop
GAMMA           = 0.99
LR              = 2.5e-4

LAMBDA = 0.99

In [4]:
def make_env():
    def _thunk():
        return gym.make("CartPole-v1")
    return _thunk



In [5]:
# we need two different model one for actor and one for critics

class Actor(nn.Module):
    def __init__(self,obs):
        super().__init__()

        self.sequential = nn.Sequential(
            nn.Linear(obs, 32), nn.ReLU(),
            nn.Linear(32,env.action_space.n), nn.Softmax(dim = -1)
        )

    def forward(self,x):
        return self.sequential(x)


class Critic(nn.Module):
    def __init__(self,obs):
        super().__init__()

        self.sequential = nn.Sequential(
            nn.Linear(obs, 32), nn.ReLU(),
            nn.Linear(32,1)
        )

    def forward(self,x):
        return self.sequential(x).squeeze(-1)


        

In [11]:
vec_env = SyncVectorEnv([make_env() for _ in range(N_ENVS)])
vec_env.observation_space.shape[0]

8

In [15]:
class Agent:
    def __init__(self):

        dummy_env = gym.make('CartPole-v1')
        self.obs_n = dummy_env.observation_space.shape[0]
        self.actor = Actor(self.obs_n)
        self.critic = Critic(self.obs_n)
        #self.env = gym.make('CartPole-v1')

        self.actor_optim = torch.optim.Adam(self.actor.parameters(), lr = LR)
        self.critic_optim = torch.optim.Adam(self.critic.parameters(), lr = LR)

        self.vec_env = SyncVectorEnv([make_env() for _ in range(N_ENVS)])


    def train(self):

        obs, _ = self.vec_env.reset()

        for i in range(1,TOTAL_UPDATES +1):

            #storage for one roll out
            obs_buf = []
            act_buf = []
            rew_buf = []
            done_buf = []
            val_buf = []

            #steps per roll out.
            for _ in range(ROLLOUT_STEPS):

                
                obs_t = torch.tensor(obs,dtype = torch.float32)
                
                logits = self.actor(obs_t)
                dist = torch.distributions.Categorical(logits=logits)
                actions = dist.sample()
                actions_np = actions.cpu().numpy()
                next_obs, rewards, terminate, truncated, _ = self.vec_env.step(actions_np)
                
                done = np.logical_or(terminate, truncated)
                # store
                obs_buf.append(obs_t)
                act_buf.append(actions)
                rew_buf.append(torch.tensor(rewards, dtype=torch.float32))
                done_buf.append(torch.tensor(done, dtype=torch.float32))
                val_buf.append(self.critic(obs_t).detach())  # detach: no grad during collection


                obs = next_obs

            #now we have our buff, time to optimize

            #find next val
            with torch.no_grad():
                next_val = self.critic(torch.tensor(obs, dtype=torch.float32))


            # here you can use gae or even simpler by just n step TD(n) return
            advantages = [torch.zeros_like(rew_buf[0]) for _ in range(ROLLOUT_STEPS)]  # shape: (T, N_ENVS)
            gae = torch.zeros_like(rew_buf[0])      # shape: (N_ENVS,)
            
            for t in reversed(range(ROLLOUT_STEPS)):
                mask = 1.0 - done_buf[t]
                delta = rew_buf[t] + GAMMA * next_val * mask - val_buf[t]
                gae = delta + GAMMA * LAMBDA * mask * gae
                advantages[t] = gae
                next_val = val_buf[t]

            advantages = torch.stack(advantages)

            returns = advantages + torch.stack(val_buf)  # TD(λ) target
        

                        # ---- flatten batch (T * N_ENVS) ----
            B, T = N_ENVS, ROLLOUT_STEPS
            obs_batch   = torch.cat(obs_buf)           # (T*B, obs_dim)
            act_batch   = torch.cat(act_buf)           # (T*B,)
            adv_batch   = advantages.view(-1)
            ret_batch   = returns.view(-1) # G

            
            # compute batch loss
            #actor
            logit = self.actor(obs_batch)
            dist = torch.distributions.Categorical(logits= logit)
            log_p = dist.log_prob(act_batch)
            entropy = dist.entropy().mean()

            actor_loss  = -(log_p * adv_batch.detach()).mean()

            critic_loss = 0.5 * (ret_batch - self.critic(obs_batch)).pow(2).mean()


            loss = actor_loss + critic_loss - 0.01 * entropy


            self.critic_optim.zero_grad()
            self.actor_optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(list(self.actor.parameters()) + list(self.critic.parameters()), 0.5)
            self.critic_optim.step()
            self.actor_optim.step()
    


            if i % 100 == 0:
                avg_return = ret_batch.view(T, B)[0].mean().item()
                print(f'Update {i:4d}, Actor loss {actor_loss:.3f}, '
                      f'Critic loss {critic_loss:.3f}, Entropy {entropy:.3f}, '
                      f'Avg 1‑step return {avg_return:.2f}')
        

        

        

        

In [16]:
agent = Agent()

In [17]:
agent.train()

Update  100, Actor loss 1.960, Critic loss 4.901, Entropy 0.688, Avg 1‑step return 4.49
Update  200, Actor loss 1.833, Critic loss 4.554, Entropy 0.688, Avg 1‑step return 4.75
Update  300, Actor loss 2.024, Critic loss 5.397, Entropy 0.688, Avg 1‑step return 5.38
Update  400, Actor loss 1.434, Critic loss 3.649, Entropy 0.686, Avg 1‑step return 4.27
Update  500, Actor loss 1.394, Critic loss 4.431, Entropy 0.684, Avg 1‑step return 5.06
Update  600, Actor loss 1.766, Critic loss 4.954, Entropy 0.682, Avg 1‑step return 4.95
Update  700, Actor loss 1.649, Critic loss 5.459, Entropy 0.680, Avg 1‑step return 6.26
Update  800, Actor loss 1.833, Critic loss 6.260, Entropy 0.673, Avg 1‑step return 5.70
Update  900, Actor loss 2.132, Critic loss 7.936, Entropy 0.671, Avg 1‑step return 8.29
Update 1000, Actor loss 0.623, Critic loss 6.185, Entropy 0.665, Avg 1‑step return 5.77
